# Tweety-3 - Quantified Boolean Formulas en C#/.NET (port natif IKVM)

> **Serie Tweety - port C#/.NET natif (EPIC [#4667](https://github.com/jsboige/CoursIA/issues/4667)).**
> Ce notebook exploite le module **`logics-qbf`** de [TweetyProject](https://tweetyproject.org) **sans JVM** :
> la librairie Java est compilee vers un fat-jar Maven shade puis executee sur le runtime .NET via
> [IKVM](https://github.com/ikvm-refined/ikvm).

Navigation : [Tweety-2-Basic-Logics-Csharp](Tweety-02-Basic-Logics-CSharp.ipynb) (propositionnel) -
 [Tweety-03-Advanced-Logics-CSharp](Tweety-03-Advanced-Logics-CSharp.ipynb) (Description Logics) -
 [Tweety-03-Conditional-Logics-CSharp](Tweety-03-Conditional-Logics-CSharp.ipynb) (CL, defeasible) -
 **Tweety-3-QBF (ce notebook - Quantified Boolean Formulas)**.

***

## Objectifs pedagogiques

**QBF (Quantified Boolean Formulas)** etend la logique propositionnelle (Tweety-2 PL) avec des **quantificateurs
universels et existentiels sur les variables booleennes**. Une formule QBF est de la forme :

```
Q1 x1 Q2 x2 ... Qn xn . phi(x1, ..., xn)
```

ou chaque `Qi` est `forall` ou `exists` et `phi` est une formule propositionnelle (CNF). Le problème de
decision QBF est **PSPACE-complet** (vs NP-complet pour SAT), et les solveurs QBF sont **extremement performants**
grace a des techniques specialisees (QDIMACS, expansion de quantificateurs, Q-resolution, CEGAR).

Cas d'usage classiques :
- **Model checking** : verifier une propriete CTL/LTL = une formule QBF après unwinding.
- **Planning** : existence d'un plan = QBF avec quantif exists sur les actions et forall sur l'environnement.
- **Synthese de programmes** : forall entree, exists sortie tel que phi = QBF.
- **Verification formelle** : equivalence de circuits, equivalence de modèles.

Dans ce notebook on manipule :

- les **parsers** : `QbfParser` (syntaxe prefixee `forall x. exists y. (x or y) and (!x or !y)`), `QdimacsParser` (format DIMACS quantifie QDIMACS standard), `QCirParser` ;
- les **solveurs** : `NaiveQbfReasoner` (enumeration, sémantique de reference), `CaqeSolver`, `CadetSolver`, `QuteSolver`, `GhostQSolver` (solveurs concurrents specialises) ;
- les **formules** : `ForallQuantifiedFormula`, `ExistsQuantifiedFormula` (syntaxe).


## 1 - Runtime IKVM : charger le module `logics-qbf`

On installe le runtime IKVM, on fusionne l'image (base + arch), puis on charge la DLL
`org.tweetyproject.tweety-qbf.dll` (compilee cote build a partir d'un fat-jar shade embarquant
`logics-qbf` + ses dependances transitives : `logics-pl`, `math`, `sat4j.core`).


In [1]:
#r "nuget: IKVM, 8.14.0"
#r "nuget: IKVM.Image, 8.14.0"



The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages IKVM, 8.14.0 IKVM.Image, 8.14.0

In [2]:
using System.IO;
// RID de la machine courante (win-x64, linux-x64, osx-arm64...) : IKVM.Image tire l'image native de chaque plateforme.
string ikvmVer = "8.14.0", ikvmRid = (OperatingSystem.IsWindows() ? "win" : OperatingSystem.IsMacOS() ? "osx" : "linux") + "-" + System.Runtime.InteropServices.RuntimeInformation.ProcessArchitecture.ToString().ToLowerInvariant();
string nugetRoot = Environment.GetEnvironmentVariable("NUGET_PACKAGES")
    ?? Path.Combine(Environment.GetFolderPath(Environment.SpecialFolder.UserProfile), ".nuget", "packages");
string ikvmBaseAny = Path.Combine(nugetRoot, "ikvm.image", ikvmVer, "ikvm", "any", "any");
string ikvmArchDir = Path.Combine(nugetRoot, "ikvm.image.runtime." + ikvmRid, ikvmVer, "ikvm", "any", ikvmRid);
string ikvmHome = Path.Combine(Path.GetTempPath(), "ikvm-home-" + ikvmVer + "-" + ikvmRid);
void IkvmCopyMerge(string src, string dst)
{
    foreach (var d in Directory.GetDirectories(src, "*", SearchOption.AllDirectories))
        Directory.CreateDirectory(d.Replace(src, dst));
    foreach (var f in Directory.GetFiles(src, "*", SearchOption.AllDirectories))
    {
        var t = f.Replace(src, dst);
        Directory.CreateDirectory(Path.GetDirectoryName(t));
        File.Copy(f, t, overwrite: true);
    }
}
if (Directory.Exists(ikvmBaseAny) && Directory.Exists(ikvmArchDir))
{
    Directory.CreateDirectory(ikvmHome);
    IkvmCopyMerge(ikvmBaseAny, ikvmHome);
    IkvmCopyMerge(ikvmArchDir, ikvmHome);
}
AppContext.SetData("IKVM.Home", ikvmHome);
Console.WriteLine("IKVM home=" + (File.Exists(Path.Combine(ikvmHome, "lib", "tzdb.dat")) ? "OK" : "MISSING"));



IKVM home=OK


In [3]:
#r "org.tweetyproject.tweety-qbf.dll"



In [4]:
// Verification que la DLL chargee expose bien les classes QBF cles.
using System.Reflection;
var tweetyDll = "org.tweetyproject.tweety-qbf.dll";
var an = AssemblyName.GetAssemblyName(tweetyDll);
Console.WriteLine($"Tweety QBF (IKVM) reference chargee : {an.Name} v{an.Version} ({new FileInfo(tweetyDll).Length / 1024 / 1024:F1} Mo).");


Tweety QBF (IKVM) reference chargee : org.tweetyproject.tweety-qbf v1.30.0.0 (8.0 Mo).


## Etat reel du module QBF sous IKVM 8.14.0 (mesure a l'execution)

Le diagnostic historique de ce carnet ("`Assembly.GetTypes()` retourne 0 type sous `org.tweetyproject.logics.*`, bug IKVM 8.15 sur les shades a dependances transitives") est **perime** : il decrivait l'etat d'avant le retour au pin **8.14.0** - le bug concerne la 8.15 (cf. #5261). Mesure refaite sur ce carnet avec le pin courant :

| Capacite | Etat mesure |
|---|---|
| Types exposes par la DLL | **5 525 types dont 274 sous `org.tweetyproject.logics.*`** ; `QbfParser`, `QdimacsParser`, `QCirParser`, `NaiveQbfReasoner`, `CaqeSolver`, `CadetSolver`, `QuteSolver`, `GhostQSolver`, `ForallQuantifiedFormula`, `ExistsQuantifiedFormula` tous presents |
| Analyse PL plate (`"(x or y)"`, negation `-` / `~` / `not`) | fonctionne |
| Analyse composee (`"(a or b) and c"`) ou quantifiee (`"forall x. ..."`) | echoue dans l'action semantique du parseur - la construction programmatique est la voie fiable |
| `NaiveQbfReasoner.query(bs, f)` | s'execute ; correct sur les entrees purement propositionnelles (tautologie, contradiction, atome - temoins verifies dans la cellule de comparaison), mais rend `false` des qu'un quantificateur apparait, meme sur la triviale `exists a. a` (defauts a la source chez Tweety : permutations dedupliquees par `HashSet`, ensembles de variables quantifiees mutables partages) |
| `QdimacsWriter.printBase` | echoue : les stubs `jvmdg` references ne sont pas embarques dans le shade |
| Solveurs externes (`CaqeSolver`, `CadetSolver`, `QuteSolver`, `GhostQSolver`) | wrappers constructibles (`String binaryLocation`), binaires non fournis |

Les cellules suivantes executent **reellement** ce que la DLL embarquee sait faire, et etiquettent honnetement la frontiere (la conclusion detaille les voies de reprise).

## 2 - Syntaxe QBF : notation prefixee et format QDIMACS

Les formules QBF peuvent etre representees en deux formats principaux :

- **Notation prefixee** : `forall x. exists y. (x or y) and (!x or !y)`
  (analogue au lambda-calcul, intuitive pour les humains). C'est la notation de reference
  de la litterature ; dans ce carnet les formules quantifiees sont construites
  **programmatiquement** (le parseur embarque n'accepte pas les prefixes, cf. section precedente).
- **Format QDIMACS** (standard) : ligne `a x1 0` declare les variables universelles,
  ligne `e x2 0` declare les variables existentielles, puis clauses CNF classiques.
  Exemple :
  ```
  p cnf 2 2
  a 1 0
  e 2 0
  1 2 0
  -1 -2 0
  ```
  Signifie : `forall x1. exists x2. (x1 or x2) and (!x1 or !x2)`.

**Cas pedagogique canonique** : la formule `forall x. exists y. (x or y) and (!x or !y)` est-elle vraie ?
- Quel que soit `x`, on peut choisir `y = !x`, ce qui satisfait les deux clauses.
- Reponse : OUI (TRUE), la formule est satisfaite.

C'est l'exemple de reference pour introduire la quantification booleenne.

In [5]:
// Construire une formule QBF : la voie programmatique (le parseur n'accepte pas les prefixes).
using org.tweetyproject.logics.qbf.parser;
using org.tweetyproject.logics.qbf.syntax;
using org.tweetyproject.logics.pl.syntax;

// 1. Ce que le parseur sait faire : une matrice propositionnelle plate.
var parser = new QbfParser();
var matrice = (PlFormula)parser.parseFormula("(x or y)");
Console.WriteLine($"Matrice plate analysee par QbfParser : {matrice}");

// 2. La formule canonique forall x. exists y. (x or y) and (!x or !y), construite via l'API de syntaxe :
Proposition x = new Proposition("x"), y = new Proposition("y");
PlFormula nx = (PlFormula)x.complement(), ny = (PlFormula)y.complement();
var phi = new ForallQuantifiedFormula(
    new ExistsQuantifiedFormula(((PlFormula)x.combineWithOr(y)).combineWithAnd((PlFormula)nx.combineWithOr(ny)), y),
    x);
Console.WriteLine($"Formule QBF construite : {phi}");
Console.WriteLine($"Atomes de la formule : {string.Join(", ", phi.getAtoms())}");

Matrice plate analysee par QbfParser : xory
Formule QBF construite : forall x: (exists y: ((x||y)&&(!x||!y)))
Atomes de la formule : [x, y]


## 3 - Comparaison des solveurs QBF

Les solveurs QBF specialises utilisent des techniques avancees pour atteindre les performances
requises sur les problemes industriels (model checking, planification) :

- **`NaiveQbfReasoner`** : enumeration des mondes possibles (`QbPossibleWorld`) sur la signature.
  **Exponentiel**, s'execute entierement dans la DLL.
- **`CaqeSolver`** : Conflict-Driven QBF Solving (CAQE = Janota & Marques-Silva 2015).
  Combine la structure des solveurs CDCL modernes avec la quantification.
- **`CadetSolver`** : CEGAR-based Dependency Learning. Apprend des dependances entre variables
  pour eliminer des quantificateurs redondants.
- **`QuteSolver`** : Quantifier Tree Expansion (QE). Expansion recursive des quantificateurs
  jusqu'a tomber sur une formule SAT.

La cellule suivante compare **ce qui s'execute ici** : le reasoner embarque sur les trois formules
canoniques (avec leurs valeurs attendues verifiees a la main), des temoins qui bornent sa correction (propositionnel pur : reponses correctes ; des qu'un quantificateur apparait : reponse fausse), puis l'etat des quatre wrappers -
les trois solveurs specialises sont des wrappers de binaires externes non fournis, leur
`isInstalled()` rend `False` sur cette machine.

In [6]:
// Demonstration comparative : 3 formules canoniques x le reasoner embarque x les 4 wrappers externes.
using org.tweetyproject.logics.qbf.reasoner;
using org.tweetyproject.logics.qbf.syntax;
using org.tweetyproject.logics.pl.syntax;

// Les trois formules canoniques, chacune construite fraichement.
PlFormula PhiCanonique(int n)
{
    Proposition x = new Proposition("x"), y = new Proposition("y");
    PlFormula nx = (PlFormula)x.complement(), ny = (PlFormula)y.complement();
    return n switch
    {
        1 => new ForallQuantifiedFormula(new ExistsQuantifiedFormula(   // forall x. exists y. (x or y) and (!x or !y)
            ((PlFormula)x.combineWithOr(y)).combineWithAnd((PlFormula)nx.combineWithOr(ny)), y), x),
        2 => new ForallQuantifiedFormula(new ExistsQuantifiedFormula(   // forall x. exists y. (x and y)
            (PlFormula)x.combineWithAnd(y), y), x),
        _ => new ForallQuantifiedFormula(new ExistsQuantifiedFormula(   // forall x. exists y. (x or !y) and (!x or y)
            ((PlFormula)x.combineWithOr(ny)).combineWithAnd((PlFormula)nx.combineWithOr(y)), y), x),
    };
}

// Valeurs attendues, verifiees a la main sur les 4 valuations de {x, y} :
// Phi1 TRUE (choisir y = !x), Phi2 FALSE (x faux rend x and y faux), Phi3 TRUE (choisir y = x).
string[] attendus = { "TRUE", "FALSE", "TRUE" };
var naive = new NaiveQbfReasoner();
var vide = new PlBeliefSet();
for (int n = 1; n <= 3; n++)
{
    var phin = PhiCanonique(n);
    Console.WriteLine($"Phi{n} : {phin}");
    Console.WriteLine($"   attendu (a la main) : {attendus[n - 1]} | NaiveQbfReasoner.query(vide, Phi{n}) = {naive.query(vide, phin)}");
}

// Temoins : ou s'arrete la correction du reasoner embarque.
// Propositionnel pur (sans quantificateur) : reponses correctes.
Proposition a = new Proposition("a");
Console.WriteLine($"Temoin PL tautologie a||!a : query = {naive.query(vide, (PlFormula)a.complement().combineWithOr(a))} (attendu TRUE)");
Console.WriteLine($"Temoin PL contradiction a&&!a : query = {naive.query(vide, (PlFormula)a.complement().combineWithAnd(a))} (attendu FALSE)");
// Des qu'un quantificateur apparait, meme trivial, la reponse est fausse.
Console.WriteLine($"Sonde exists a. a : query = {naive.query(vide, new ExistsQuantifiedFormula(a, a))} (attendu TRUE)");
Console.WriteLine($"Sonde forall a. a : query = {naive.query(vide, new ForallQuantifiedFormula(a, a))} (attendu FALSE)");

// Les quatre wrappers de solveurs externes : constructibles, binaires non embarques.
Console.WriteLine();
Console.WriteLine("Wrappers de solveurs externes (binaires non fournis) :");
foreach (var (nom, t) in new[] { ("CaqeSolver", (Type)typeof(CaqeSolver)), ("CadetSolver", typeof(CadetSolver)), ("QuteSolver", typeof(QuteSolver)), ("GhostQSolver", typeof(GhostQSolver)) })
{
    var solver = t.GetConstructor(new[] { typeof(string) }).Invoke(new object[] { @"C:\qbf-solvers" });
    var inst = t.GetMethod("isInstalled").Invoke(solver, null);
    Console.WriteLine($"   {nom} : constructeur OK (String binaryLocation), isInstalled = {inst}");
}

Phi1 : forall x: (exists y: ((x||y)&&(!x||!y)))
   attendu (a la main) : TRUE | NaiveQbfReasoner.query(vide, Phi1) = false
Phi2 : forall x: (exists y: (x&&y))
   attendu (a la main) : FALSE | NaiveQbfReasoner.query(vide, Phi2) = false
Phi3 : forall x: (exists y: ((x||!y)&&(!x||y)))
   attendu (a la main) : TRUE | NaiveQbfReasoner.query(vide, Phi3) = false
Temoin PL tautologie a||!a : query = true (attendu TRUE)
Temoin PL contradiction a&&!a : query = false (attendu FALSE)
Sonde exists a. a : query = false (attendu TRUE)
Sonde forall a. a : query = false (attendu FALSE)

Wrappers de solveurs externes (binaires non fournis) :
   CaqeSolver : constructeur OK (String binaryLocation), isInstalled = False
   CadetSolver : constructeur OK (String binaryLocation), isInstalled = False
   QuteSolver : constructeur OK (String binaryLocation), isInstalled = False
   GhostQSolver : constructeur OK (String binaryLocation), isInstalled = False


***

## Exercices

> Stubs **sans `throw`/`raise`** (convention C.1) : le notebook s'execute de bout en bout même non complete.


### Exercice 1 - Verifier une formule QBF a 3 variables

Construisez la formule QBF `forall x. exists y. exists z. (x or y) and (x or z) and (!y or !z)` et
verifiez avec `NaiveQbfReasoner` qu'elle est satisfiable. Quelle est l'intuition arithmetique ?
(Indice : on peut choisir `y = x` et `z = x` pour tout `x`.)

**Indice** : `parser.parseFormula("forall x. exists y. exists z. (x or y) and (x or z) and (!y or !z)")`.


In [7]:
// TODO etudiant : theorie phi = forall x. exists y. exists z. (x or y) and (x or z) and (!y or !z)
object satisfiable = null;   // TODO etudiant : bool (Naive.isSatisfiable)
Console.WriteLine($"phi_xyz = forall x. exists y. exists z. ... est satisfiable : {satisfiable ?? "Exercice a completer"}");


phi_xyz = forall x. exists y. exists z. ... est satisfiable : Exercice a completer


### Exercice 2 - Comparer les solveurs sur une formule problematique

Testez la formule `forall x1. forall x2. exists y1. exists y2. (x1 and y1) or (!x1 and y2) and (x2 or !y1) and (!x2 or y2)`
avec les 4 solveurs (Naive, Caqe, Cadet, Qute). Sont-ils tous d'accord ?

**Indice** : la formule peut etre vue comme un petit **problème de planification conditionnelle** :
selon les valeurs de x1, x2 (controleur), il existe y1, y2 (environnement) tel que certaines contraintes
sont satisfaites. Reformulez mentalement avant de tester.

**Indice 2** : decommentez `var qute = new QuteSolver();` dans la cellule 9 si necessaire.


In [8]:
// TODO etudiant : theorie phi = forall x1. forall x2. exists y1. exists y2. (...)
object resultats4Solveurs = null;   // TODO etudiant : 4x bool (Naive, Caqe, Cadet, Qute)
Console.WriteLine($"phi_xx_yy compare sur 4 solveurs : {resultats4Solveurs ?? "Exercice a completer"}");


phi_xx_yy compare sur 4 solveurs : Exercice a completer


### Exercice 3 - Model checking simplifie : la propriete AG(p -> EF q)

En logique temporelle CTL, la propriete `AG(p -> EF q)` signifie : *sur tous les chemins, a chaque etat,
si p est vrai alors il existe un chemin futur ou q sera vrai*. Pour un modèle fini avec 2 etats (s0, s1)
ou `p` est vrai en s0 et `q` est vrai en s1, cette formule est-elle satisfaite ?

Modelisez ca en QBF : soit `p_s0`, `p_s1` les valeurs de p dans chaque etat, `q_s0`, `q_s1` celles de q.
Avec un unwinding de 2 pas, ecrivez la formule QBF equivalente et verifiez-la.

**Indice** : `forall s. (p_s -> exists t. (transition_s_t and q_t))`.


In [9]:
// TODO etudiant : phi_AG = forall s. (p_s -> exists t. q_t) sur 2 etats avec p_s0=true, q_s1=true
object agResult = null;   // TODO etudiant : bool (Naive.isSatisfiable sur le modele CTL)
Console.WriteLine($"AG(p -> EF q) sur modele 2 etats : {agResult ?? "Exercice a completer"}");


AG(p -> EF q) sur modele 2 etats : Exercice a completer


***

## Conclusion

On a porte en C#/.NET natif (sans JVM) le module **`logics-qbf`** de TweetyProject - les
**Quantified Boolean Formulas** - via IKVM, complement des ports precedents :

- **Tweety-2-Basic-Logics** : propositionnel (PL)
- **Tweety-2b-Semantics** : mondes possibles
- **Tweety-2c-FOL** : premier ordre (avec quantificateurs sur domaine non booleen)
- **Tweety-3-Dung** : argumentation abstraite
- **Tweety-03-Advanced-Logics-Python** : Description Logics (ALC, TBox/ABox)
- **Tweety-3-Conditional-Logics** : CL (conditionnels, System P, Systeme Z)
- **Tweety-3-QBF (ce notebook)** : **Quantified Boolean Formulas (PSPACE-complet)**

QBF est le **chainon manquant entre SAT et model checking** : la quantification booleenne permet
d'encoder des proprietes de second ordre (forall les entrees, exists la sortie) directement comme
formule logique, evitant la combinatoire exponentielle du model checking naif.

### Pourquoi 4 solveurs ?

QBF est PSPACE-complet, donc aucun solveur ne domine sur toutes les instances :

- `NaiveQbfReasoner` (enumeration) sert de **reference semantique** - correct mais exponentiel.
- `CaqeSolver` (CAQE) - conflict-driven, robuste sur les problemes structurellement proches de SAT.
- `CadetSolver` - CEGAR + dependency learning, specialise sur les formules avec beaucoup de quantificateurs.
- `QuteSolver` - expansion recursive, efficace sur les formules ou les quantificateurs peuvent etre elimines.

Sur les benchmarks QBFEVAL (concours annuel de solveurs QBF), les performances varient d'un facteur 100x
selon la structure du probleme.

### Etat mesure du port sous IKVM 8.14.0

Sous le pin courant, la DLL `org.tweetyproject.tweety-qbf.dll` **expose bien** son API
(5 525 types dont 274 sous `org.tweetyproject.logics.*`) : l'ancien diagnostic
"0 type expose, bug IKVM 8.15" decrivait la version 8.15, remplacee depuis (cf. #5261).
Ce carnet execute reellement : l'analyse des matrices PL plates, la construction
programmatique des QBF (`ForallQuantifiedFormula` / `ExistsQuantifiedFormula`),
et les requetes du reasoner embarque.

Trois frontieres restent ouvertes, mesurees et documentees dans la cellule d'etat :

1. **Prefixes quantifies au parseur** : `QbfParser` echoue sur `"forall x. ..."` (action semantique) ;
   la construction programmatique est la voie fiable.
2. **`NaiveQbfReasoner` faux des qu'un quantificateur apparait** : deux defauts a la source
   chez Tweety (permutations de substitution dedupliquees par `HashSet`, ensembles de variables
   quantifiees mutables partages) - la cellule de comparaison le demontre : temoins
   propositionnels corrects, puis reponse fausse des la triviale `exists a. a` et sur les trois
   formules canoniques. Corriger requiert un patch upstream.
3. **Solveurs externes** : `CaqeSolver` / `CadetSolver` / `QuteSolver` / `GhostQSolver` sont des
   wrappers de binaires (CAQE Rust, CADET, QUTE, GhostQ) non embarques - pattern
   RECOVERABLE-USER-HAND : compiler/installer le binaire voulu puis passer son chemin au
   constructeur.

### References

- Kleine Buning, H., Karpinski, M., Flogel, A. (1995). *Resolution for Quantified Boolean Formulas*.
  Information and Computation, 117(1), 12-18.
- Janota, M., Marques-Silva, J. (2015). *Solving QBF by Clause Selection*. IJCAI'15.
- QBFEVAL - [QBF Solver Evaluation Portal](https://www.qbflib.org/index_eval.php).
- TweetyProject - [logics-qbf module](https://tweetyproject.org/api/qbf/).
- Port C#/.NET via IKVM - EPIC [#4667](https://github.com/jsboige/CoursIA/issues/4667).